# 01 - EDA dos Dados

Objetivos:
- analisar questões, temas e respostas
- validar qualidade dos dados
- entender distribuição de interações
- preparar insumos para DKT e LPKT


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

QUESTIONS_PATH = RAW_DIR / 'questions.json'
ANSWERS_PATH = RAW_DIR / 'answers.json'
TREE_PATH = RAW_DIR / 'tree.json'
SKILL_TO_H2_PATH = PROCESSED_DIR / 'mappings' / 'skill_to_h2.json'

print(PROJECT_ROOT)
print(QUESTIONS_PATH.exists(), ANSWERS_PATH.exists(), TREE_PATH.exists())

In [ ]:
questions = pd.read_json(QUESTIONS_PATH)
answers = pd.read_json(ANSWERS_PATH)

with open(TREE_PATH, 'r', encoding='utf-8') as f:
    tree = json.load(f)

skill_to_h2 = {}
if SKILL_TO_H2_PATH.exists():
    with open(SKILL_TO_H2_PATH, 'r', encoding='utf-8') as f:
        skill_to_h2 = json.load(f)

print('questions:', len(questions))
print('answers:', len(answers))
print('tree top-level nodes:', len(tree.get('data', {}).get('subjectNodes', [])) if isinstance(tree, dict) else len(tree))
print('skill_to_h2:', len(skill_to_h2))

## Visão geral dos dados

In [ ]:
display(questions.head())
display(answers.head())
questions.info()
answers.info()

In [ ]:
summary = {
    'n_questions': len(questions),
    'n_answers': len(answers),
    'n_users': answers['user_id'].nunique(),
    'n_question_ids_in_answers': answers['question_id'].nunique(),
    'n_skill_ids_in_answers': answers['skill_id'].nunique(),
    'n_institutions': questions['institutionName'].nunique() if 'institutionName' in questions.columns else None,
}
summary

## Qualidade dos dados

In [ ]:
nulls_answers = answers.isna().sum().sort_values(ascending=False)
nulls_questions = questions.isna().sum().sort_values(ascending=False)
display(nulls_answers)
display(nulls_questions)

In [ ]:
dup_answers = answers.duplicated().sum()
invalid_correct = (~answers['correct'].isin([0, 1])).sum() if 'correct' in answers.columns else None
negative_time_response = (answers['time_response'] < 0).sum() if 'time_response' in answers.columns else None

{
    'duplicated_answers': int(dup_answers),
    'invalid_correct_values': int(invalid_correct) if invalid_correct is not None else None,
    'negative_time_response': int(negative_time_response) if negative_time_response is not None else None,
}

## Distribuição de respostas e interações

In [ ]:
answers['correct'].value_counts(normalize=True).sort_index()

In [ ]:
user_counts = answers.groupby('user_id').size().sort_values(ascending=False)
user_counts.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(user_counts, bins=50)
plt.title('Distribuição de interações por usuário')
plt.xlabel('Número de interações')
plt.ylabel('Frequência')
plt.show()

In [ ]:
answers['time_response'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
answers_sorted = answers.sort_values(['user_id', 'timestamp']).copy()
answers_sorted['delta_t'] = answers_sorted.groupby('user_id')['timestamp'].diff().fillna(0)
answers_sorted['delta_t'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

## Enriquecimento com H2

In [ ]:
if skill_to_h2:
    answers_h2 = answers.copy()
    answers_h2['h2'] = answers_h2['skill_id'].astype(str).map(skill_to_h2)
    display(answers_h2['h2'].value_counts(dropna=False).head(20))
else:
    print('skill_to_h2.json ainda não encontrado.')

## Cruzamento questões x respostas

In [ ]:
question_cols = ['questionId', 'subjectId', 'institutionName', 'examYear']
question_cols = [c for c in question_cols if c in questions.columns]
merged = answers.merge(questions[question_cols], left_on='question_id', right_on='questionId', how='left')
display(merged.head())
merged[['question_id']].shape

## Conclusões iniciais

Preencher após executar:
- balanceamento de acertos
- sparsity por aluno e por skill
- distribuição temporal
- cobertura por H2
- potenciais riscos de ruído ou desbalanceamento
